# Building A GPT Model

In [108]:
# Imports

import torch
import torch.nn as nn
from torch.nn import functional as F

In [319]:
# Hyperparameters

batch_size = 32    # number of independent sequences processed in parallel
block_size = 16   # aka. context window
max_iters = 10000
eval_interval = 500
learning_rate = 3e-4
device = (
    torch.device('cuda') if torch.cuda.is_available() else # if available use Nvidia GPUs
    torch.device('mps') if torch.mps.is_available() else   # if available use Apple's Metal Performance Shaders
    torch.device('cpu')
)
eval_iters = 200
n_embd = 64
n_heads = 6
n_blocks = 6
dropout = 0.2

print(f"Using device: {device}")

Using device: mps


In [106]:
# download the tiny shakespeare dataset
!curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  9354k      0 --:--:-- --:--:-- --:--:-- 9390k


In [313]:
torch.manual_seed(1337) # set manual seed for reproducability

In [133]:
# Explore the dataset

print("length of dataset in characters: ", len(text))

print("\n-----\n")

print(text[:300])

print("\n-----\n")

length of dataset in characters:  1115394

-----

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us

-----



In [121]:
chars = sorted(list(set(text))) # unique chars in the data
vocab_size = len(chars) # number of unique chars

# Generate bi-directional mapping from chars to integers
chartoi = {ch:i for i,ch in enumerate(chars)}
itochar = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [chartoi[c] for c in s] # encodes a string into ints
decode = lambda l: "".join([itochar[i] for i in l]) # decodes list of ints into string of chars

In [130]:
# Inspect unique chars and vocab size
print(f"vocab_size = {vocab_size}")
print(f"\nUnique chars:")
"|".join(chars)

vocab_size = 65

Unique chars:


"\n| |!|$|&|'|,|-|.|3|:|;|?|A|B|C|D|E|F|G|H|I|J|K|L|M|N|O|P|Q|R|S|T|U|V|W|X|Y|Z|a|b|c|d|e|f|g|h|i|j|k|l|m|n|o|p|q|r|s|t|u|v|w|x|y|z"

In [127]:
# Encode the entire dataset
data = torch.tensor(encode(text), dtype=torch.long)

print(f"data.shape = {data.shape}")
print(f"data[:10] = {data[:10]}")

data.shape = torch.Size([1115394])
data[:10] = tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [134]:
# Split into train and validation sets
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

print(f"train_data.shape = {train_data.shape}")
print(f"val_data.shape = {val_data.shape}")

train_data.shape = torch.Size([1003854])
val_data.shape = torch.Size([111540])


In [315]:
# data loading
def get_batch(split, batch_size=4):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")

print(f"xb.shape = {xb.shape}")
print()
print("xb:")
print(xb)
print()
print(f"yb.shape = {yb.shape}")
print()
print("yb:")
print(yb)

xb.shape = torch.Size([4, 128])

xb:
tensor([[ 1, 40, 56, 39, 61, 52,  6,  0, 27, 56,  1, 50, 53, 57, 43,  1, 51, 47,
         52, 43,  1, 39, 56, 51,  1, 44, 53, 56, 58, 10,  1, 58, 46, 53, 59,  1,
         46, 39, 57, 58,  1, 40, 43, 39, 58,  1, 51, 43,  1, 53, 59, 58,  0, 32,
         61, 43, 50, 60, 43,  1, 57, 43, 60, 43, 56, 39, 50,  1, 58, 47, 51, 43,
         57,  6,  1, 39, 52, 42,  1, 21,  1, 46, 39, 60, 43,  1, 52, 47, 45, 46,
         58, 50, 63,  1, 57, 47, 52, 41, 43,  0, 16, 56, 43, 39, 51, 58,  1, 53,
         44,  1, 43, 52, 41, 53, 59, 52, 58, 43, 56, 57,  1,  5, 58, 61, 47, 62,
         58,  1],
        [16, 33, 23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21, 27, 10,  0, 25, 63,
          1, 46, 53, 50, 63,  1, 57, 47, 56,  6,  1, 52, 53, 52, 43,  1, 40, 43,
         58, 58, 43, 56,  1, 49, 52, 53, 61, 57,  1, 58, 46, 39, 52,  1, 63, 53,
         59,  0, 20, 53, 61,  1, 21,  1, 46, 39, 60, 43,  1, 43, 60, 43, 56,  1,
         50, 53, 60, 43, 42,  1, 58, 46, 43,  1, 50, 4

In [320]:
class Head(nn.Module):
    """ Single head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)  # (B,T,head_size)
        q = self.query(x) # (B,T,head_size)
        v = self.value(x) # (B,T,head_size)

        wei = (q @ k.transpose(-2, -1))/self.head_size**0.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        wei = self.dropout(wei)

        out = wei @ v # (B,T,head_size)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x is of shape (B,T,C) where C=n_embd
        out = torch.cat([h(x) for h in self.heads], dim=-1) #(B,T, num_heads * head_size)
        out = self.dropout(self.proj(out)) #(B,T,C)
        return out

class FeedForward(nn.Module):
    """ simple multi-layer perceptron: linear tranform + relu """
    
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=True),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd, bias=True), # projection layer before adding to residual pathway
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # x.shape = (B, T, C) where C = n_embd
        out = self.net(x) # (B, T, C)
        return out
        
class Block(nn.Module):
    """ single pair of self-attentions heads + feed-forward layer (with residual connections) """
    
    def __init__(self, n_embd, num_heads):
        super().__init__()
        self.self_attention_heads = MultiHeadAttention(num_heads, n_embd//num_heads)
        self.feed_forward = FeedForward(n_embd)
        self.layer_norm_1 = nn.LayerNorm(n_embd)
        self.layer_norm_2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x.shape is (B,T,C) where C = n_embd
        x = x + self.self_attention_heads(self.layer_norm_1(x)) # (B,T,C)
        x = x + self.feed_forward(self.layer_norm_2(x))      # (B,T,C)
        return x

class CharacterLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_blocks)])
        self.final_layer_norm = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        
        # idx and targets are both (B,T) tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C) where C = n_emdb
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb    # (B,T,C)
        x = self.blocks(x)       # (B,T,C)
        x = self.final_layer_norm(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_crop = idx[:, -block_size:]
            # get predictions
            logits, loss = self.forward(idx_crop)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [321]:
model = CharacterLanguageModel().to(device)

In [322]:
# generate using the model:

print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist()))


j!;EinQEtWCq
CC,jmFigEuePhuMMUdIsnVrO.k:UYU.oJEHqCAkRwyN;zYNjIOPERwghJNSgUBRBgBMuV-bGV&SJgV-HupOwHGl
,RGhvIQycNegsUWJse:qnseKkubTYbnFoADNOksgsJb?NGIus-bZ,
QwseHftdqswSdsEEoc.Hr:BTekGe;HiNuzuMoNNAe
muD-;XXaNJ$:XY:ihnGBKhUwGNggINYfF-tzJHyfMUh-WfxAcnWuGcY.gggeIM'lBNsEiT:BsEZkJ.JADuqJPwTDJmgTtqq
zZ;SNik:AcHcuM&Ukrz?Z
Gf:T
Gkk&jElNTqCMU?wAEAiYJJjkbIsnXHSdZgJdP;-cK.NunBGU
XYLJKlkBe'iv.:sMONhKovgdIgVpcseNmBeT
yJX-VEyw
XKMfJu?GjQOKm:qY.OsoCAqkA;sATMgCJJRpVvnUy;zJnIdeyqzkny
 eHaUhhj&IJTPqeUykuQE-'bkE?fN-


In [323]:
# create a pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [324]:
# Better estimate of true loss by averaging across multiple batches sampled
# from train and val datasets

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

@torch.no_grad()
def estimate_loss_v2():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        X, Y = get_batch(split, batch_size*eval_iters)
        logits, loss = model(X, Y)
        out[split] = loss
    model.train()
    return out

In [325]:
# The training loop: evaluate batch of examples --> compute gradients --> update weights in the dir that minimizes loss --> repeat 

for e in range(max_iters):

    if e % eval_interval == 0:
        losses = estimate_loss_v2()
        print(f"Step {e:4d}: train loss = {losses["train"].item():.4f} | val loss = {losses["val"].item():.4f}")
    
    xb, yb = get_batch('train', batch_size)
    
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Step    0: train loss = 4.3451 | val loss = 4.3459
Step  500: train loss = 2.4379 | val loss = 2.4404
Step 1000: train loss = 2.2846 | val loss = 2.3020
Step 1500: train loss = 2.1878 | val loss = 2.2052
Step 2000: train loss = 2.1192 | val loss = 2.1406
Step 2500: train loss = 2.0577 | val loss = 2.1012
Step 3000: train loss = 2.0213 | val loss = 2.0689
Step 3500: train loss = 1.9763 | val loss = 2.0410
Step 4000: train loss = 1.9552 | val loss = 2.0273
Step 4500: train loss = 1.9193 | val loss = 2.0030
Step 5000: train loss = 1.8972 | val loss = 1.9837
Step 5500: train loss = 1.8940 | val loss = 1.9733
Step 6000: train loss = 1.8665 | val loss = 1.9659
Step 6500: train loss = 1.8482 | val loss = 1.9579
Step 7000: train loss = 1.8368 | val loss = 1.9581
Step 7500: train loss = 1.8201 | val loss = 1.9354
Step 8000: train loss = 1.8058 | val loss = 1.9340
Step 8500: train loss = 1.7987 | val loss = 1.9223
Step 9000: train loss = 1.7872 | val loss = 1.9272
Step 9500: train loss = 1.7804 

In [326]:
# generate using the model post-training:

print(decode(model.generate(torch.zeros((1, 1), dtype=torch.long, device=device), 500)[0].tolist()))


CRICHBERWARD:
Axer no nor this not Rucharder: to cither,
But is lagge:
ThenBer donest the worroed,
A mom dey.

HERDWARD LELIZABEY:
And we senvole an fork a drom
Licer owly theg many'd haplLar da no apposains!
By gor come an know to has'd the carciom,?
D; you give son shalf baut did you, nester see my than he prarl of that ved heary defush,
Do;
Tho ou dom, Ony nink therberowes, in frepend doble
Casterond at dike, rejust
Of their me grice at it nath here his pect:
Mo stall stell, no se wome.

MENE
